## Lab 5: AgentCore Evaluations - Online Evaluation for Customer Support Agent

### Overview

This lab demonstrates how to use AgentCore Evaluations to continuously monitor your production customer support agent from Lab 4. You'll configure online evaluation to automatically assess agent performance in real-time as customers interact with it.

**Workshop Journey:**

- **Lab 1 (Done):** Create Agent Prototype - Built a functional customer support agent
- **Lab 2 (Done):** Enhance with Memory - Added conversation context and personalization
- **Lab 3 (Done):** Scale with Gateway & Identity - Shared tools across agents securely
- **Lab 4 (Done):** Deploy to Production - Used AgentCore Runtime with observability
- **Lab 5 (Current):** Evaluate Agent Performance - Monitor quality with online evaluations
- **Lab 6:** Build User Interface - Create a customer-facing application

### What You'll Learn

You'll configure online evaluation with built-in evaluators, generate test interactions, and analyze quality metrics through AgentCore Observability dashboards to improve agent performance.

### Online Evaluation Overview

Online evaluation continuously monitors deployed agents in production, unlike on-demand evaluation which analyzes specific selected interactions. It consists of three components: session sampling with configurable rules, multiple evaluation methods (built-in or custom evaluators), and monitoring through dashboards with quality trends and low-scoring session investigation.

Since your agent runs on AgentCore Runtime, AgentCore Observability automatically instruments the code and provides comprehensive logs and traces using [OTEL](https://opentelemetry.io/) instrumentation.

### Prerequisites

Complete Lab 4 to have the customer support agent deployed. You'll need AWS account access to Amazon Bedrock AgentCore with Evaluations permissions.

### Architecture
<div style="text-align:left">
    <img src="images/architecture_lab5_evaluation.png" width="75%"/>
</div>

*Online evaluation automatically monitors agent interactions, applies evaluators based on sampling rules, and outputs results to CloudWatch for analysis.*

### Step 1: Import Required Libraries and Initialize Clients

In [1]:
from bedrock_agentcore_starter_toolkit import Evaluation, Runtime
import json
import uuid
from pathlib import Path
from boto3.session import Session
from IPython.display import Markdown, display
from lab_helpers.utils import get_ssm_parameter, get_or_create_cognito_pool

In [2]:
boto_session = Session()
region = boto_session.region_name
print(f"Region: {region}")

Region: us-west-2


In [3]:
eval_client = Evaluation(region=region)
runtime_client = Runtime()

### Step 2: Retrieve Agent Information from Lab 4

Retrieve the customer support agent ARN from SSM Parameter Store where it was saved during Lab 4 deployment.

In [4]:
try:
    # Get agent ARN from SSM parameter store (saved in Lab 4)
    agent_arn = get_ssm_parameter("/app/customersupport/agentcore/runtime_arn")
    
    # Extract agent ID from ARN
    agent_id = agent_arn.split(":")[-1].split("/")[-1]
    
    # Set runtime client config path
    runtime_client._config_path = Path.cwd() / ".bedrock_agentcore.yaml"
    
    print("Agent ID:", agent_id)
    print("Agent ARN:", agent_arn)
except Exception as e:
    raise Exception(f"""Missing agent information from Lab 4. Please run lab-04-agentcore-runtime.ipynb first. Error: {str(e)}""")

Agent ID: customer_support_agent-y14e0mGWVx
Agent ARN: arn:aws:bedrock-agentcore:us-west-2:061264750636:runtime/customer_support_agent-y14e0mGWVx


### Step 3: Create Online Evaluation Configuration

Now let's create an online evaluation configuration for our customer support agent. We'll use built-in evaluators to assess different aspects of agent performance:

- **Builtin.GoalSuccessRate** - Measures how well the agent achieves user goals
- **Builtin.Correctness** - Evaluates factual accuracy of responses
- **Builtin.ToolSelectionAccuracy** - Evaluates appropriate tool selection

We'll set the sampling rate to 100% for demonstration purposes, but in production you might use a lower rate (e.g., 10-20%) based on your traffic volume.

In [5]:
response = eval_client.create_online_config(
    agent_id=agent_id,
    config_name="customer_support_agent_eval",
    sampling_rate=100,  # Evaluate 100% of sessions for demo
    evaluator_list=[
        "Builtin.GoalSuccessRate", 
        "Builtin.Correctness",
        "Builtin.ToolSelectionAccuracy"
    ],
    config_description="Customer support agent online evaluation",
    auto_create_execution_role=True
)

print("Online evaluation configuration created successfully!")
print(f"Configuration ID: {response['onlineEvaluationConfigId']}")

Creating online evaluation config: customer_support_agent_eval for agent: customer_support_agent-y14e0mGWVx
Configuration: sampling_rate=100.0%, evaluators=['Builtin.GoalSuccessRate', 'Builtin.Correctness', 'Builtin.ToolSelectionAccuracy']
Creating online evaluation config: customer_support_agent_eval for agent: customer_support_agent-y14e0mGWVx
Auto-creating execution role for config: customer_support_agent_eval
Getting or creating evaluation execution role for config: customer_support_agent_eval
Using AWS region: us-west-2, account ID: 061264750636
Role name: AgentCoreEvalsSDK-us-west-2-d04ba7b68b
Role doesn't exist, creating new evaluation execution role: AgentCoreEvalsSDK-us-west-2-d04ba7b68b
Creating IAM role: AgentCoreEvalsSDK-us-west-2-d04ba7b68b
✓ Role created: arn:aws:iam::061264750636:role/AgentCoreEvalsSDK-us-west-2-d04ba7b68b
✓ Execution policy attached: AgentCoreEvaluationPolicy-us-west-2-d04ba7b68b
Waiting for IAM role propagation...
Role creation complete and ready for u

✅ Online evaluation configuration created!

Online evaluation configuration created successfully!
Configuration ID: customer_support_agent_eval-wC1RZU2Kc2


### Step 4: Verify Configuration Status

Verify the evaluation configuration is properly created and enabled by retrieving its details.

In [6]:
config_details = eval_client.get_online_config(config_id=response['onlineEvaluationConfigId'])
print("Configuration Details:")
print(json.dumps(config_details, indent=2, default=str))

Configuration Details:
{
  "ResponseMetadata": {
    "RequestId": "a5a7819c-81c2-4b50-a634-ae98a1a024b9",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Tue, 03 Mar 2026 14:19:36 GMT",
      "content-type": "application/json",
      "content-length": "1052",
      "connection": "keep-alive",
      "x-amzn-requestid": "a5a7819c-81c2-4b50-a634-ae98a1a024b9",
      "x-amzn-remapped-x-amzn-requestid": "a5a7819c-81c2-4b50-a634-ae98a1a024b9",
      "x-amzn-remapped-content-length": "1052",
      "x-amzn-remapped-connection": "keep-alive",
      "x-cache": "Miss from cloudfront",
      "via": "1.1 38789cdd14ddea5c4c609cb0e6656396.cloudfront.net (CloudFront)",
      "x-amz-cf-id": "EC9EV7E77oRyOzIrHJTM3_7iTt1NJ0UNxCqXbuJ915GEhTo-8CRlUQ==",
      "x-amz-apigw-id": "Zpoe8FHqPHcEoOA=",
      "x-amzn-trace-id": "Root=1-69a6edf8-157ec3e30237efad028fd8e9",
      "x-amz-cf-pop": "HIO52-P4",
      "x-amzn-remapped-date": "Tue, 03 Mar 2026 14:19:36 GMT"
    },
    "RetryAttempts": 0
  }

### Step 5: Generate Test Interactions

Invoke the customer support agent with various queries to generate traces for evaluation. Different test scenarios will demonstrate how the evaluators assess agent performance.

In [7]:
# Get authentication token
access_token = get_or_create_cognito_pool(refresh_token=True)
print(f"Access token obtained: {access_token['bearer_token'][:20]}...")

def invoke_agent_runtime(prompt, session_id=None):
    """Invoke the agent runtime using starter toolkit"""
    if not session_id:
        session_id = str(uuid.uuid4())
    
    response = runtime_client.invoke(
        payload={"prompt": prompt},
        session_id=session_id,
        bearer_token=access_token['bearer_token']
    )
    
    return response, session_id

Access token obtained: eyJraWQiOiJRK3loMnJr...


#### Test Scenario 1: Product Information Query

In [8]:
session1 = str(uuid.uuid4())
response, _ = invoke_agent_runtime(
    "I need information about the Gaming Console Pro. What are its specifications and price?",
    session1
)
print("Customer Query: Product information request")
display(Markdown(response["response"].replace('\\n', '\n')))

Using JWT authentication


Customer Query: Product information request


"I understand you're looking for information about the Gaming Console Pro. Unfortunately, I'm unable to retrieve the technical specifications for this specific model at the moment. 

However, I can help you in a few ways:

1. I can connect you with our technical support team who can provide detailed specifications, compatibility information, and answer any technical questions you may have

2. If you're interested in purchasing or learning about pricing, I can check current pricing and availability for you

3. If you have a specific question about gaming performance, connectivity, or features, I can try to help with that

Would you like me to contact technical support on your behalf, or would you prefer to share more details about what specific information you're looking for (like gaming performance specs, controller compatibility, storage options, etc.)?"

#### Test Scenario 2: Technical Support Request

In [9]:
session2 = str(uuid.uuid4())
response, _ = invoke_agent_runtime(
    "My laptop won't start up. Can you help me troubleshoot this issue?",
    session2
)
print("Customer Query: Technical support request")
display(Markdown(response["response"].replace('\\n', '\n')))

Using JWT authentication


Customer Query: Technical support request


"  

I can help you troubleshoot your laptop startup issue! To provide the most effective assistance, could you please share a few more details?

**Quick Diagnostic Questions:**

1.  **When you try to start your laptop, what exactly happens?**  
 *   Does it power on but the screen remains black?  
    *   Do you see any lights (power, battery, indicator lights) come on?  
    *   Do you hear any fans spinning or hard drive activity?  
    *   Does it beep at all?  

2.  **Have you made any recent changes?**  
    *   Did you install any new software or updates recently?  
    *   Did you add or remove any hardware (like RAM, a second battery, etc.)?  
    *   Was the laptop dropped or exposed to liquid?  

3.  **What operating system is installed?** (Windows, macOS, Linux, etc.)

4.  **Do you have any recovery options?**  
    *   Does it boot into Safe Mode?  
    *   Can you access the BIOS/UEFI settings by pressing a key during startup (often F1, F2, Delete, or Esc)?  

**Based on your answers, here are some common troubleshooting steps we can try:**

*   **Basic Power Check:** Ensure the power adapter is plugged in securely and the battery isn't completely dead. Try a hard reset: unplug power, remove the battery (if removable), hold the power button for 30 seconds, reassemble, and try again.
*   **Safe Mode/BIOS Access:** Try booting into Safe Mode (usually by pressing F8, Shift+Restart, or another key specific to your OS during startup) or access the BIOS/UEFI settings. This can help determine if it's a software or hardware issue.
*   **External Monitor:** Connect the laptop to an external monitor or TV to see if the display is the problem.
*   **Minimal Hardware Check:** If possible, try booting with minimal hardware (remove any extra monitors, peripherals, extra RAM sticks if applicable - though be cautious with RAM removal).

**Please let me know the details from the questions above, and we'll go from there!** I'm ready to help you get your ThinkPad back up and running."

#### Test Scenario 3: Return Policy Inquiry

In [10]:
session3 = str(uuid.uuid4())
response, _ = invoke_agent_runtime(
    "I bought a smartphone last week but it's not working properly. What's your return policy?",
    session3
)
print("Customer Query: Return policy inquiry")
display(Markdown(response["response"].replace('\\n', '\n')))

Using JWT authentication


Customer Query: Return policy inquiry


" 

I can help you with that! To provide you with the most accurate return policy information, could you please tell me what type of smartphone you purchased? For example, is it an iPhone, Samsung Galaxy, or another brand/model?

Once you share this information, I'll get the exact return policy details for your specific smartphone type."

#### Test Scenario 4: Complex Multi-Tool Query

In [11]:
session4 = str(uuid.uuid4())
response, _ = invoke_agent_runtime(
    "I need help with my Gaming Console Pro. First, can you tell me about its warranty? Then I need technical support for connection issues.",
    session4
)
print("Customer Query: Complex multi-tool request")
display(Markdown(response["response"].replace('\\n', '\n')))

Using JWT authentication


Customer Query: Complex multi-tool request


" 

I can help you with both your Gaming Console Pro warranty information and technical support for connection issues. Let me assist you step by step.

First, let me check the warranty information for your Gaming Console Pro. I'll need the serial number to look this up. Could you please provide the serial number for your Gaming Console Pro?

Once I have that, I can check your warranty status. Then I can also help you with the connection issues you're experiencing.

Would you like to proceed with checking your warranty status first? If so, please share the serial number for your Gaming Console Pro."

#### Test Scenario 5: General Capability Query

In [12]:
session5 = str(uuid.uuid4())
response, _ = invoke_agent_runtime(
    "What kind of support can you provide? List all your available tools and capabilities.",
    session5
)
print("Customer Query: Capability inquiry")
display(Markdown(response["response"].replace('\\n', '\n')))

Using JWT authentication


Customer Query: Capability inquiry


" 

Hello! I'm your dedicated customer support assistant for our electronics e-commerce platform. I can help with a wide range of technical and support inquiries regarding our products and services. Here's a comprehensive overview of my capabilities and available tools:

## Available Tools and Capabilities:

### **Product Information**
1. **`get_product_info(product_type)`** - Provides detailed technical specifications for any electronics product we sell
   - Product categories: laptops, smartphones, headphones, monitors, tablets, gaming consoles, accessories
   - Includes warranty info, features, compatibility details, and technical specs

### **Return & Warranty Policies**
2. **`get_return_policy(product_category)`** - Accesses our current return policy for specific product categories
   - Covers timeframes, conditions, and procedures for returns and exchanges

### **Technical Support**
3. **`get_technical_support(issue_description)`** - Provides troubleshooting guidance and solutions for technical problems
   - Diagnoses issues and offers step-by-step solutions
   - Works for hardware, software, connectivity, and performance problems

### **Warranty Verification**
4. **`LambdaUsingSDK___check_warranty_status(serial_number, customer_email)`** - Checks warranty status using product serial number
   - Verifies remaining warranty period
   - Can optionally send verification to customer email

### **Research & Information**
5. **`LambdaUsingSDK___web_search(keywords, region, max_results)`** - Searches for updated technical documentation and information
   - Accesses current technical specs, compatibility guides, and troubleshooting resources
   - Supports various regions and can return multiple results

## What I Can Help You With:

**✅ Product Selection & Research**
- Compare specifications between different models
- Find compatible accessories and peripherals
- Check system requirements for software/hardware

**✅ Technical Support & Troubleshooting**
- Diagnose hardware or software issues
- Step-by-step repair guidance
- Connectivity and performance optimization
- Driver and firmware update information

**✅ Warranty & Returns**
- Verify warranty coverage and expiration dates
- Process return requests and understand policies
- Get information about repair services

**✅ Compatibility Verification**
- Check Linux compatibility for laptops and peripherals
- Verify gaming console compatibility
- Assess low-latency requirements for gaming audio equipment

**✅ Competitive Gaming Setup**
- Recommend low-latency audio equipment for FPS gaming
- Optimize gaming console settings
- Troubleshoot gaming performance issues

**✅ Programming & Development Tools**
- Verify Linux compatibility for laptops
- Check development tool requirements
- Provide programming environment setup guidance

Would you like me to help you with any specific product, technical issue, or compatibility question right now? I can look up detailed specifications for any electronics product you're interested in, troubleshoot a particular problem, or help you understand our return and warranty policies."

### Step 6: Monitor Evaluation Results

Monitor evaluation results through the AgentCore Observability console. Results may take a few minutes to appear as the system processes traces and applies evaluators.

#### Accessing the Dashboard

1. Navigate to the [AgentCore Observability console](https://console.aws.amazon.com/cloudwatch/home#gen-ai-observability/agent-core/agents)
2. Find your customer support agent in the agents list
3. Click on the `DEFAULT` endpoint to view evaluation metrics
4. Look for the evaluation scores in the traces and sessions views

#### What You'll See

The dashboard will show:
- **Goal Success Rate**: How well the agent achieves customer objectives
- **Correctness**: Accuracy of information provided
- **Tool Selection Accuracy**: Appropriate tool choices for queries

![Online Evaluation Dashboard](images/online_evaluations_dashboard.png)

*Evaluation metrics displayed in the AgentCore Observability dashboard*

### Step 7: Understanding Evaluation Metrics

**Goal Success Rate** measures whether the agent successfully addresses the customer's primary intent. High scores indicate effective problem-solving; low scores suggest unmet needs, incomplete responses, or misunderstood requests.

**Correctness** evaluates factual accuracy of responses. High scores indicate accurate and reliable information; low scores suggest incorrect facts, outdated information, or misleading guidance.

**Tool Selection Accuracy** evaluates whether the agent chooses appropriate tools for each task. High scores indicate proper tool selection; low scores suggest wrong tools, unnecessary calls, or missing tool usage.

### Step 8: Analyzing Results and Next Steps

**For Low Goal Success Rates:** Refine the agent's system prompt, improve tool descriptions and parameters, and add specific training examples.

**For Low Correctness Scores:** Update the knowledge base with current information, improve fact-checking mechanisms, and review tool responses.

**For Tool-Related Issues:** Refine tool parameter schemas, improve tool selection logic, and enhance tool documentation.

**Continuous Monitoring:** Set up CloudWatch alarms for evaluation metrics, create dashboards for trend analysis, and implement automated alerts for quality degradation.

### Step 9: Clean Up (Optional)

Disable the online evaluation configuration if needed by uncommenting the code below.

In [13]:
# Uncomment the following lines if you want to disable the evaluation configuration
# eval_client.delete_online_config(config_id=response['onlineEvaluationConfigId'])
# print("Online evaluation configuration disabled")

### Congratulations! 🎉

You have successfully completed **Lab 5: AgentCore Evaluations - Online Evaluation!**

### What You Accomplished

You configured automatic continuous online evaluation for your customer support agent with built-in evaluators assessing Goal Success Rate (customer satisfaction and problem resolution), Correctness (factual accuracy), and Tool Selection Accuracy (proper tool usage). Evaluation results are integrated with AgentCore Observability dashboards for real-time insights.

**Key Benefits:** Proactive quality assurance catches issues before customer impact, data-driven optimization guides improvements, production confidence through performance monitoring at scale, and continuous learning identifies patterns and opportunities.

**Next Steps:** Monitor your evaluation dashboard regularly, set up CloudWatch alarms for quality thresholds, use insights to iteratively improve your agent, and consider adding custom evaluators for domain-specific metrics.

### Next Up: [Lab 6: Build User Interface →](lab-06-frontend.ipynb)

Complete the customer experience by building a user-friendly web interface for customers to interact with your quality-monitored agent.

Your customer support agent is now production-ready with comprehensive quality monitoring! 🚀